In [ ]:
from pathlib import Path
import json

import fitz
import numpy as np
import tiktoken
from sentence_transformers import SentenceTransformer


In [ ]:
FILES_DIR = Path("Files")
MODEL_NAME = "all-MiniLM-L6-v2"
ENCODING_NAME = "cl100k_base"
CHUNK_SIZE = 60
TOP_K = 5
INCLUDE_CHECKPOINTS = False


In [ ]:
def canonical_pdf_key(path: Path) -> str:
    parts = [part for part in path.parts if part != ".ipynb_checkpoints"]
    normalized = Path(*parts)
    if normalized.name.endswith("-checkpoint.pdf"):
        normalized = normalized.with_name(
            normalized.name.replace("-checkpoint.pdf", ".pdf")
        )
    return normalized.as_posix().lower()


def iter_pdf_files(folder: Path, include_checkpoints: bool = False):
    seen = set()
    for pdf_path in sorted(folder.rglob("*.pdf")):
        if not include_checkpoints:
            if ".ipynb_checkpoints" in pdf_path.parts:
                continue
            if pdf_path.name.endswith("-checkpoint.pdf"):
                continue

        key = canonical_pdf_key(pdf_path)
        if key in seen:
            continue

        seen.add(key)
        yield pdf_path


def extract_page_texts(pdf_path: Path):
    doc = fitz.open(str(pdf_path))
    try:
        if doc.needs_pass:
            return "encrypted", []

        page_texts = []
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                page_texts.append((page_number, text))

        if page_texts:
            return "text", page_texts
        return "none", []
    finally:
        doc.close()


def chunk_page_text(tokenizer, page_number: int, text: str, chunk_size: int):
    token_ids = tokenizer.encode(text)
    for start in range(0, len(token_ids), chunk_size):
        chunk_ids = token_ids[start:start + chunk_size]
        chunk_text = tokenizer.decode(chunk_ids).strip()
        if chunk_text:
            yield {
                "page": page_number,
                "text": chunk_text,
            }


In [ ]:
tokenizer = tiktoken.get_encoding(ENCODING_NAME)
model = SentenceTransformer(MODEL_NAME)

doc_records = []
chunk_records = []

for doc_index, pdf_path in enumerate(iter_pdf_files(FILES_DIR, INCLUDE_CHECKPOINTS)):
    method, page_texts = extract_page_texts(pdf_path)
    doc_id = f"doc_{doc_index:05d}"
    char_count = sum(len(text) for _, text in page_texts)

    doc_records.append({
        "doc_id": doc_id,
        "pdf_path": pdf_path.as_posix(),
        "method": method,
        "char_count": char_count,
    })

    if method != "text":
        continue

    chunk_id = 0
    for page_number, page_text in page_texts:
        for chunk in chunk_page_text(tokenizer, page_number, page_text, CHUNK_SIZE):
            chunk_records.append({
                "doc_id": doc_id,
                "pdf_path": pdf_path.as_posix(),
                "page": chunk["page"],
                "chunk_id": chunk_id,
                "text": chunk["text"],
            })
            chunk_id += 1

print(f"Documents scanned: {len(doc_records)}")
print(f"Text chunks created: {len(chunk_records)}")
print("Methods:", {
    "text": sum(record["method"] == "text" for record in doc_records),
    "none": sum(record["method"] == "none" for record in doc_records),
    "encrypted": sum(record["method"] == "encrypted" for record in doc_records),
})


In [ ]:
chunk_texts = [record["text"] for record in chunk_records]
chunk_vectors = model.encode(chunk_texts, normalize_embeddings=True)
chunk_vectors = np.asarray(chunk_vectors, dtype=np.float32)

docs_by_id = {record["doc_id"]: record for record in doc_records}

print("Embedding matrix shape:", chunk_vectors.shape)
print("Example document:", doc_records[0] if doc_records else "No documents found")


In [ ]:
vector_store_dir = Path("vector_store_notebook")
vector_store_dir.mkdir(exist_ok=True)

with (vector_store_dir / "docs.jsonl").open("w", encoding="utf-8") as f:
    for record in doc_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with (vector_store_dir / "chunks.jsonl").open("w", encoding="utf-8") as f:
    for record in chunk_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

np.save(vector_store_dir / "vectors.npy", chunk_vectors)

print(f"Saved notebook vector store to {vector_store_dir}")


In [ ]:
query = "cold air is coming through the window"

query_vector = model.encode([query], normalize_embeddings=True)[0]
scores = chunk_vectors @ query_vector

top_k = min(TOP_K, len(chunk_records))
top_indices = np.argsort(scores)[-top_k:][::-1]

print(f'Query: {query!r}')
print(f"Top {top_k} matches:")
print()

for rank, index in enumerate(top_indices, start=1):
    chunk = chunk_records[int(index)]
    doc = docs_by_id[chunk["doc_id"]]
    preview = " ".join(chunk["text"].split())

    print(f"Match #{rank}")
    print(f"Score: {float(scores[index]):.4f}")
    print(f"Document: {Path(doc['pdf_path']).name}")
    print(f"Path: {doc['pdf_path']}")
    print(f"Page: {chunk['page']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print("Best matching text:")
    print(preview[:700])
    print()
